In [7]:
class User:
    def __init__(self, username, is_admin=False):
        self.username = username
        self.is_admin = is_admin


def admin_required(func):
    def wrapper(*args, **kwargs):
        user = kwargs.get("user") or (args[0] if args else None)
        if user is None or not getattr(user, "is_admin", False):
            raise PermissionError("Admin access required")
        return func(*args, **kwargs)
    return wrapper


@admin_required
def delete_user_account(user, target_username):
    print(f"{user.username} deleted {target_username}'s account")


admin = User("alice", is_admin=True)
normal_user = User("bob", is_admin=False)

delete_user_account(admin, "charlie")      # OK
#delete_user_account(normal_user, "david")  # Raises PermissionError

alice deleted charlie's account


In [8]:
class User:
    def __init__(self, username, is_authenticated=True):
        self.username = username
        self.is_authenticated = is_authenticated


class Request:
    def __init__(self, method, path, user):
        self.method = method
        self.path = path
        self.user = user


def authenticate(func):
    def wrapper(request, *args, **kwargs):
        if not request.user.is_authenticated:
            print("[AUTH] user is NOT authenticated, skipping handler")
            return None
        print("[AUTH] user is authenticated")
        return func(request, *args, **kwargs)
    return wrapper


def log_request(func):
    def wrapper(request, *args, **kwargs):
        print(f"[LOG] {request.method} {request.path} by {request.user.username}")
        return func(request, *args, **kwargs)
    return wrapper


def track_execution(func):
    def wrapper(*args, **kwargs):
        print(f"[EXEC] starting {func.__name__}")
        result = func(*args, **kwargs)
        print(f"[EXEC] finished {func.__name__}")
        return result
    return wrapper


@track_execution
@log_request
@authenticate
def get_profile(request, user_id):
    print(f"[HANDLER] getting profile for user_id={user_id}")
    return {"id": user_id, "name": "Alice"}


# demo
req_ok = Request("GET", "/api/profile/1", User("bob", True))
print(get_profile(req_ok, 1))

req_bad = Request("GET", "/api/profile/1", User("anon", False))
print(get_profile(req_bad, 1))

[EXEC] starting wrapper
[LOG] GET /api/profile/1 by bob
[AUTH] user is authenticated
[HANDLER] getting profile for user_id=1
[EXEC] finished wrapper
{'id': 1, 'name': 'Alice'}
[EXEC] starting wrapper
[LOG] GET /api/profile/1 by anon
[AUTH] user is NOT authenticated, skipping handler
[EXEC] finished wrapper
None
